# Pothole Video Pipeline

Combined final notebook:
1. Bronze: raw video ingest
2. Silver: frame extraction
3. Gold: frame classification
4. Gold-enriched: positive pothole frames with metadata
5. Handoff to future location stage

In [0]:
%pip install torch torchvision pillow opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.9/542.9 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
spark.sql("USE CATALOG trendsmarketplace")
spark.sql("USE SCHEMA pothole_project")
print("Spark ready")

Spark ready


In [0]:
video_path = "/Volumes/trendsmarketplace/pothole_project/raw_videos/videoplayback.mp4"
run_id = "daily_run_01"

model_path = "/Volumes/trendsmarketplace/pothole_project/model_artifacts/model_v3_traced.pt"
config_path = "/Volumes/trendsmarketplace/pothole_project/model_artifacts/config.json"

frames_base_dir = f"/Volumes/trendsmarketplace/pothole_project/frames/{run_id}"
bronze_table = f"bronze_videos_{run_id}"
silver_table = f"silver_frames_{run_id}"
gold_table = f"gold_pothole_frames_{run_id}"
gold_positive_table = f"gold_pothole_positive_frames_{run_id}"

In [0]:
bronze_videos_df = (
    spark.read.format("binaryFile")
    .load(video_path)
    .withColumn("video_id", F.sha2(F.col("path"), 256))
    .withColumn("file_name", F.element_at(F.split(F.col("path"), "/"), -1))
    .withColumn("ingest_time", F.current_timestamp())
    .select(
        "video_id", "file_name", "path",
        F.col("modificationTime").alias("modification_time"),
        "length", "content", "ingest_time"
    )
)
display(bronze_videos_df.select("video_id", "file_name", "path", "modification_time", "length", "ingest_time"))

video_id,file_name,path,modification_time,length,ingest_time
39c15a6796ac6730081a188b4752614ea1c4ea2a6f7fd62a878ed6ac6e6b89a1,videoplayback.mp4,dbfs:/Volumes/trendsmarketplace/pothole_project/raw_videos/videoplayback.mp4,2026-04-17T18:23:45.000Z,624934602,2026-04-18T00:12:10.889Z


In [0]:
bronze_videos_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
print(f"Saved Bronze table: {bronze_table}")

Saved Bronze table: bronze_videos_daily_run_01


In [0]:
import cv2
import os
import uuid

def extract_frames(video_path, output_dir, every_n_seconds=1):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        raise ValueError("Could not read FPS from video.")
    frame_interval = int(fps * every_n_seconds)
    frame_index = 0
    saved_rows = []
    os.makedirs(output_dir, exist_ok=True)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_index % frame_interval == 0:
            frame_id = str(uuid.uuid4())
            timestamp_sec = frame_index / fps
            file_name = f"{frame_id}.jpg"
            frame_path = f"{output_dir}/{file_name}"
            cv2.imwrite(frame_path, frame)
            saved_rows.append(Row(
                frame_id=frame_id,
                frame_path=frame_path,
                frame_index=frame_index,
                timestamp_sec=timestamp_sec
            ))
        frame_index += 1

    cap.release()
    return spark.createDataFrame(saved_rows)

In [0]:
silver_frames_df = extract_frames(video_path, frames_base_dir, every_n_seconds=1)
display(silver_frames_df.limit(10))

frame_id,frame_path,frame_index,timestamp_sec
f6f265fe-c12b-4c7c-ad93-dfe373451fcd,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/f6f265fe-c12b-4c7c-ad93-dfe373451fcd.jpg,0,0.0
934a2d1e-3519-4184-b0b4-67ca0aea65c9,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/934a2d1e-3519-4184-b0b4-67ca0aea65c9.jpg,30,1.0
3d494b27-cfb0-4f5d-b19d-6bdde72a7f18,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/3d494b27-cfb0-4f5d-b19d-6bdde72a7f18.jpg,60,2.0
c421c3cc-30c5-4d54-90fe-3f61dcf33b9f,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/c421c3cc-30c5-4d54-90fe-3f61dcf33b9f.jpg,90,3.0
599c9a56-079c-4404-a7a7-c52ef6642009,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/599c9a56-079c-4404-a7a7-c52ef6642009.jpg,120,4.0
a41ad06c-18b8-4d3e-8836-5997cc0fd39c,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/a41ad06c-18b8-4d3e-8836-5997cc0fd39c.jpg,150,5.0
07a480ed-2507-49aa-944f-a4bbfb48fdf9,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/07a480ed-2507-49aa-944f-a4bbfb48fdf9.jpg,180,6.0
6b7c857c-cf32-4fb8-8a41-4405c263a20b,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/6b7c857c-cf32-4fb8-8a41-4405c263a20b.jpg,210,7.0
3b0c6524-40dc-413c-8fc1-0f296022578f,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/3b0c6524-40dc-413c-8fc1-0f296022578f.jpg,240,8.0
ad2eba23-2b62-4041-9f66-9b7d155b7cae,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ad2eba23-2b62-4041-9f66-9b7d155b7cae.jpg,270,9.0


In [0]:
silver_frames_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
print(f"Saved Silver table: {silver_table}")

Saved Silver table: silver_frames_daily_run_01


In [0]:
silver_frames_df = spark.table(silver_table)
display(silver_frames_df.limit(5))

frame_id,frame_path,frame_index,timestamp_sec
f6f265fe-c12b-4c7c-ad93-dfe373451fcd,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/f6f265fe-c12b-4c7c-ad93-dfe373451fcd.jpg,0,0.0
934a2d1e-3519-4184-b0b4-67ca0aea65c9,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/934a2d1e-3519-4184-b0b4-67ca0aea65c9.jpg,30,1.0
3d494b27-cfb0-4f5d-b19d-6bdde72a7f18,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/3d494b27-cfb0-4f5d-b19d-6bdde72a7f18.jpg,60,2.0
c421c3cc-30c5-4d54-90fe-3f61dcf33b9f,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/c421c3cc-30c5-4d54-90fe-3f61dcf33b9f.jpg,90,3.0
599c9a56-079c-4404-a7a7-c52ef6642009,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/599c9a56-079c-4404-a7a7-c52ef6642009.jpg,120,4.0


In [0]:
import json
import torch

with open(config_path, "r") as f:
    config = json.load(f)

classifier = torch.jit.load(model_path, map_location="cpu")
classifier.eval()

print("Classifier loaded")
print(config)

Classifier loaded
{'arch': 'resnet18', 'num_classes': 2, 'class_to_idx': {'non_pothole': 0, 'pothole': 1}, 'pothole_idx': 1, 'default_threshold': 0.5, 'input_size': 224, 'normalize_mean': [0.485, 0.456, 0.406], 'normalize_std': [0.229, 0.224, 0.225], 'training_sources': ['RDD2020 (relabelled D40 pothole / D00+D10 as non_pothole)', 'Sovit Simplex pothole dataset', 'Pothole_Segmentation_YOLOv8 (70% of positive images)'], 'external_test_results': {'neha_normal_pothole_dataset @ threshold=0.50': {'accuracy': 0.8824, 'recall_pothole': 0.9064, 'precision': 0.8649, 'f1': 0.8852, 'fn_rate': 0.0936, 'fp_rate': 0.1416}}, 'notes': 'Binary pothole classifier. Input: RGB road image. Output: pothole probability.'}


In [0]:
from torchvision import transforms

input_size = config["input_size"]
mean = config["normalize_mean"]
std = config["normalize_std"]

transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

pothole_idx = int(config["pothole_idx"])
threshold = float(config["default_threshold"])
print("Transform ready")

Transform ready


In [0]:
from PIL import Image

def classify_frame(frame_path):
    img = Image.open(frame_path).convert("RGB")
    x = transform(img).unsqueeze(0)
    with torch.no_grad():
        logits = classifier(x)
        probs = torch.softmax(logits, dim=1)
    pothole_prob = probs[0, pothole_idx].item()
    pred_class = torch.argmax(probs, dim=1).item()
    is_pothole = pothole_prob >= threshold
    return {
        "frame_path": frame_path,
        "pothole_prob": pothole_prob,
        "pred_class": pred_class,
        "is_pothole": is_pothole
    }

In [0]:
def classify_frames_in_chunks(silver_frames_df, chunk_size=100):
    all_paths = [row["frame_path"] for row in silver_frames_df.select("frame_path").collect()]
    all_results = []
    for start in range(0, len(all_paths), chunk_size):
        batch_paths = all_paths[start:start + chunk_size]
        batch_results = [classify_frame(path) for path in batch_paths]
        all_results.extend(batch_results)
        print(f"finished {start + len(batch_paths)} / {len(all_paths)}")
    return spark.createDataFrame(all_results)

In [0]:
sample_paths = [row["frame_path"] for row in silver_frames_df.limit(10).select("frame_path").collect()]
sample_results = [classify_frame(path) for path in sample_paths]
sample_df = spark.createDataFrame(sample_results)
display(sample_df)
display(sample_df.filter("is_pothole = true"))

frame_path,is_pothole,pothole_prob,pred_class
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/f6f265fe-c12b-4c7c-ad93-dfe373451fcd.jpg,false,0.032426364719867706,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/934a2d1e-3519-4184-b0b4-67ca0aea65c9.jpg,false,0.47482144832611084,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/3d494b27-cfb0-4f5d-b19d-6bdde72a7f18.jpg,false,0.007337698247283697,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/c421c3cc-30c5-4d54-90fe-3f61dcf33b9f.jpg,false,0.13613472878932953,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/599c9a56-079c-4404-a7a7-c52ef6642009.jpg,false,0.13408590853214264,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/a41ad06c-18b8-4d3e-8836-5997cc0fd39c.jpg,false,0.16571646928787231,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/07a480ed-2507-49aa-944f-a4bbfb48fdf9.jpg,false,0.026187527924776077,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/6b7c857c-cf32-4fb8-8a41-4405c263a20b.jpg,false,0.0010758353164419532,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/3b0c6524-40dc-413c-8fc1-0f296022578f.jpg,false,0.04275143891572952,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ad2eba23-2b62-4041-9f66-9b7d155b7cae.jpg,false,0.06360156089067459,0


frame_path,is_pothole,pothole_prob,pred_class


In [0]:
gold_full_df = classify_frames_in_chunks(silver_frames_df, chunk_size=100)
display(gold_full_df)

finished 100 / 2834
finished 200 / 2834
finished 300 / 2834
finished 400 / 2834
finished 500 / 2834
finished 600 / 2834
finished 700 / 2834
finished 800 / 2834
finished 900 / 2834
finished 1000 / 2834
finished 1100 / 2834
finished 1200 / 2834
finished 1300 / 2834
finished 1400 / 2834
finished 1500 / 2834
finished 1600 / 2834
finished 1700 / 2834
finished 1800 / 2834
finished 1900 / 2834
finished 2000 / 2834
finished 2100 / 2834
finished 2200 / 2834
finished 2300 / 2834
finished 2400 / 2834
finished 2500 / 2834
finished 2600 / 2834
finished 2700 / 2834
finished 2800 / 2834
finished 2834 / 2834


frame_path,is_pothole,pothole_prob,pred_class
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/2e44148b-074f-4118-932e-8b68ddc2e987.jpg,false,0.009225983172655106,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/b9323a15-bc50-49c8-9315-797535a98423.jpg,false,0.10092011094093323,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/b0c0b2c6-bf95-4d99-889f-c95ec72d4844.jpg,false,0.05702260881662369,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/c75182d1-3577-4992-94d2-52c944427578.jpg,false,0.43012726306915283,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ba77bcc7-4ac8-455e-bda0-82526bd0e550.jpg,false,8.67689959704876E-4,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/952c9069-b295-4bd0-8e81-f9d97a4c9c1f.jpg,false,0.015093638561666012,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/98dbab87-f69d-454d-9778-5c2d52698cba.jpg,false,0.08656761050224304,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ca2b96a7-ab14-49bc-87c6-0dac2d91358a.jpg,false,0.045113205909729004,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/2a8f39a9-6dd7-4176-8487-f3fa5e45f7a8.jpg,false,5.772683653049171E-4,0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/0c69bedc-c1d5-4d3c-8f34-d1627e871230.jpg,false,0.03303684666752815,0


In [0]:
positive_frames_df = gold_full_df.filter("is_pothole = true")
display(positive_frames_df)
print("Positive frame count:", positive_frames_df.count())

frame_path,is_pothole,pothole_prob,pred_class
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ff2ac550-75eb-4ad7-bc77-acd5505eca3c.jpg,true,0.9832955002784729,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/0e3a2ed2-5145-4960-a898-424d85ead7c6.jpg,true,0.8610879182815552,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/988a5f53-12ee-4ae7-bae7-e72f0fc586da.jpg,true,0.9482334852218628,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/d459c472-7637-477a-9488-2e800992e74a.jpg,true,0.5636512637138367,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ba6b2bda-fd6d-4a86-bae9-c879080f177a.jpg,true,0.7745388150215149,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/80ccf91d-20e6-44bd-82cc-ec07c9232eb3.jpg,true,0.8275828957557678,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/bcc634fd-becc-4635-a9a2-569940e2c12d.jpg,true,0.9993984699249268,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/415c8faf-aec1-4a22-87db-46dce3a7b1c3.jpg,true,0.9729649424552917,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/21927aad-0f15-4192-a31a-7a38df7e590a.jpg,true,0.7366986870765686,1
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/32f09543-5931-4cff-b004-346e0e183022.jpg,true,0.981805145740509,1


Positive frame count: 244


In [0]:
gold_enriched_df = (
    gold_full_df.alias("g")
    .join(silver_frames_df.alias("s"), on="frame_path", how="left")
)

positive_frames_enriched_df = gold_enriched_df.filter("is_pothole = true")
display(positive_frames_enriched_df)

frame_path,is_pothole,pothole_prob,pred_class,frame_id,frame_index,timestamp_sec
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ff2ac550-75eb-4ad7-bc77-acd5505eca3c.jpg,true,0.9832955002784729,1,ff2ac550-75eb-4ad7-bc77-acd5505eca3c,57090,1903.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/0e3a2ed2-5145-4960-a898-424d85ead7c6.jpg,true,0.8610879182815552,1,0e3a2ed2-5145-4960-a898-424d85ead7c6,57150,1905.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/988a5f53-12ee-4ae7-bae7-e72f0fc586da.jpg,true,0.9482334852218628,1,988a5f53-12ee-4ae7-bae7-e72f0fc586da,58020,1934.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/d459c472-7637-477a-9488-2e800992e74a.jpg,true,0.5636512637138367,1,d459c472-7637-477a-9488-2e800992e74a,58380,1946.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ba6b2bda-fd6d-4a86-bae9-c879080f177a.jpg,true,0.7745388150215149,1,ba6b2bda-fd6d-4a86-bae9-c879080f177a,59550,1985.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/80ccf91d-20e6-44bd-82cc-ec07c9232eb3.jpg,true,0.8275828957557678,1,80ccf91d-20e6-44bd-82cc-ec07c9232eb3,61470,2049.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/bcc634fd-becc-4635-a9a2-569940e2c12d.jpg,true,0.9993984699249268,1,bcc634fd-becc-4635-a9a2-569940e2c12d,61500,2050.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/415c8faf-aec1-4a22-87db-46dce3a7b1c3.jpg,true,0.9729649424552917,1,415c8faf-aec1-4a22-87db-46dce3a7b1c3,61530,2051.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/21927aad-0f15-4192-a31a-7a38df7e590a.jpg,true,0.7366986870765686,1,21927aad-0f15-4192-a31a-7a38df7e590a,61590,2053.0
/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/32f09543-5931-4cff-b004-346e0e183022.jpg,true,0.981805145740509,1,32f09543-5931-4cff-b004-346e0e183022,62340,2078.0


In [0]:
gold_full_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)
positive_frames_enriched_df.write.format("delta").mode("overwrite").saveAsTable(gold_positive_table)

print(f"Saved Gold table: {gold_table}")
print(f"Saved positive Gold table: {gold_positive_table}")

Saved Gold table: gold_pothole_frames_daily_run_01
Saved positive Gold table: gold_pothole_positive_frames_daily_run_01


In [0]:
summary_df = positive_frames_enriched_df.select(
    "frame_id", "frame_path", "frame_index", "timestamp_sec", "pothole_prob"
).orderBy(F.desc("pothole_prob"))

display(summary_df)

frame_id,frame_path,frame_index,timestamp_sec,pothole_prob
4f5aa874-c2a8-44b8-ae3e-a2ac467b0087,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/4f5aa874-c2a8-44b8-ae3e-a2ac467b0087.jpg,900,30.0,0.9999545812606812
d94c5726-e877-43dc-b8a3-8b4144d02cc9,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/d94c5726-e877-43dc-b8a3-8b4144d02cc9.jpg,22500,750.0,0.9997124075889587
69314049-91de-42ff-95af-fc92937b1db3,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/69314049-91de-42ff-95af-fc92937b1db3.jpg,68010,2267.0,0.9996405839920044
bcc634fd-becc-4635-a9a2-569940e2c12d,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/bcc634fd-becc-4635-a9a2-569940e2c12d.jpg,61500,2050.0,0.9993984699249268
8d98ee15-9854-4c72-a52f-104d64786680,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/8d98ee15-9854-4c72-a52f-104d64786680.jpg,9720,324.0,0.9992509484291077
adf6439b-b720-4469-a23d-30de5c618132,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/adf6439b-b720-4469-a23d-30de5c618132.jpg,55050,1835.0,0.9983545541763306
1be8e180-4ad9-413f-a9a7-5015f0adc292,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/1be8e180-4ad9-413f-a9a7-5015f0adc292.jpg,55230,1841.0,0.9969885945320129
aad99ecc-2bb0-4f32-8bfe-3fe7ae1eacb0,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/aad99ecc-2bb0-4f32-8bfe-3fe7ae1eacb0.jpg,23040,768.0,0.9958339929580688
cc9a2937-5e4f-4685-ab61-5f8f51e8833f,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/cc9a2937-5e4f-4685-ab61-5f8f51e8833f.jpg,53880,1796.0,0.995587944984436
b82f4732-8d58-4f00-b891-cbc765d2c69a,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/b82f4732-8d58-4f00-b891-cbc765d2c69a.jpg,56190,1873.0,0.9947169423103333


In [0]:
positive_for_location_df = positive_frames_enriched_df.select(
    "frame_id", "frame_path", "frame_index", "timestamp_sec", "pothole_prob"
)
display(positive_for_location_df)

frame_id,frame_path,frame_index,timestamp_sec,pothole_prob
ff2ac550-75eb-4ad7-bc77-acd5505eca3c,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ff2ac550-75eb-4ad7-bc77-acd5505eca3c.jpg,57090,1903.0,0.9832955002784729
0e3a2ed2-5145-4960-a898-424d85ead7c6,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/0e3a2ed2-5145-4960-a898-424d85ead7c6.jpg,57150,1905.0,0.8610879182815552
988a5f53-12ee-4ae7-bae7-e72f0fc586da,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/988a5f53-12ee-4ae7-bae7-e72f0fc586da.jpg,58020,1934.0,0.9482334852218628
d459c472-7637-477a-9488-2e800992e74a,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/d459c472-7637-477a-9488-2e800992e74a.jpg,58380,1946.0,0.5636512637138367
ba6b2bda-fd6d-4a86-bae9-c879080f177a,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/ba6b2bda-fd6d-4a86-bae9-c879080f177a.jpg,59550,1985.0,0.7745388150215149
80ccf91d-20e6-44bd-82cc-ec07c9232eb3,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/80ccf91d-20e6-44bd-82cc-ec07c9232eb3.jpg,61470,2049.0,0.8275828957557678
bcc634fd-becc-4635-a9a2-569940e2c12d,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/bcc634fd-becc-4635-a9a2-569940e2c12d.jpg,61500,2050.0,0.9993984699249268
415c8faf-aec1-4a22-87db-46dce3a7b1c3,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/415c8faf-aec1-4a22-87db-46dce3a7b1c3.jpg,61530,2051.0,0.9729649424552917
21927aad-0f15-4192-a31a-7a38df7e590a,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/21927aad-0f15-4192-a31a-7a38df7e590a.jpg,61590,2053.0,0.7366986870765686
32f09543-5931-4cff-b004-346e0e183022,/Volumes/trendsmarketplace/pothole_project/frames/daily_run_01/32f09543-5931-4cff-b004-346e0e183022.jpg,62340,2078.0,0.981805145740509


In [0]:
def build_silver(video_path, run_id, every_n_seconds=1):
    output_dir = f"/Volumes/trendsmarketplace/pothole_project/frames/{run_id}"
    silver_table_local = f"silver_frames_{run_id}"
    silver_df = extract_frames(video_path, output_dir, every_n_seconds=every_n_seconds)
    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_local)
    return spark.table(silver_table_local), silver_table_local

def build_gold(silver_frames_df, run_id, chunk_size=100):
    gold_table_local = f"gold_pothole_frames_{run_id}"
    gold_positive_table_local = f"gold_pothole_positive_frames_{run_id}"
    gold_df = classify_frames_in_chunks(silver_frames_df, chunk_size=chunk_size)
    gold_enriched_local = gold_df.alias("g").join(silver_frames_df.alias("s"), on="frame_path", how="left")
    positive_local = gold_enriched_local.filter("is_pothole = true")
    gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table_local)
    positive_local.write.format("delta").mode("overwrite").saveAsTable(gold_positive_table_local)
    return gold_df, positive_local, gold_table_local, gold_positive_table_local

In [0]:
# Daily-run block:
# video_path = "/Volumes/trendsmarketplace/pothole_project/raw_videos/new_video.mp4"
# run_id = "2026_04_17_video_b"
# silver_frames_df, silver_table = build_silver(video_path, run_id, every_n_seconds=1)
# gold_full_df, positive_frames_enriched_df, gold_table, gold_positive_table = build_gold(silver_frames_df, run_id, chunk_size=100)
# display(positive_frames_enriched_df)